# L2 step 1b · Soft labels from the new 7B, for option (b) (GPU T4 ×2)

Task **L2** (roadmap), the distilled 3B for objective **O4**. Decided 8 Nov: train two 3Bs, **(b)** with soft labels and **(c)** with the options rerun, and keep the better one by the rule in roadmap L2, fixed before any run. This notebook makes (b)'s soft labels.

**What a soft label is.** When the 7B reads an answer, it gives every possible next token a probability. A normal label keeps only the right token; a soft label keeps the 7B's whole spread, for example "12: 60%, 13: 20%, 11: 10%". The 3B then learns from the label and from how sure the 7B was (Hinton et al., 2015). The 7B and the 3B share Qwen2.5's tokenizer, so the spreads line up token by token.

**What it does**
- For each checkable training question (counting, road counting, disaster type, bearing bodies, land use, relational reasoning), the 7B (`AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP`, fp16, both T4s) reads the images, the question and its correct answer (the dataset's label) in one pass. No text is generated.
- At each answer token, and at the end-of-answer token, it keeps the 20 most likely tokens and their log-probabilities, plus the log-probability of the true token.
- The images reach the 7B at the 3B's training size (at most 768 × 28 × 28 pixels), so teacher and student see the same pictures.
- Left out: captions and restoration advice (long free text; the 3B learns them from the label only), and every question on the 501 after-photos held out to choose the winner. That leaves **38,271** questions (1,962 held out).
- Self-check on every question: the decoded answer tokens must equal the label's text. Above 0.5% mismatches, the notebook stops.

**Expectation, fixed before the run (8 Nov):** the 7B gives the label's tokens a high probability on most questions, since it was trained on these labels: its top-1 token equals the label's on most answer tokens, lowest on the numbers of counting answers. No number was predicted. Time: the smoke test measures it; about 5 h was estimated.

**How to run on Kaggle** (account 3, where training (b) runs next and reads this output)
1. Settings: GPU **T4 x2**, internet ON. No secret: the 7B's repository is public (a `HF_TOKEN` secret only speeds up the 16 GB download).
2. Input: `abrarmohammedtanzim/disasterm3-mirror`.
3. Run All once with `SMOKE_TEST = True` in Cell 2 (16 questions; 15–20 minutes with the download). Check Cell 5's sample and Cell 6's projected hours. Then set it to `False` and use Save & Run All.
4. It stops cleanly at 11 h. To continue, attach that version's output and run again: questions already done are skipped.
5. Outputs in `/kaggle/working/L2S1b/`: `shards/*.npz` (the soft labels), `README.md`, `config.json`, `by_task.csv`.


In [1]:
# ── Cell 1: Environment (Kaggle's own torch; transformers only if too old for Qwen2.5-VL) ──
import importlib.metadata as _md
import subprocess
import sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return "0"

def _older(have, want):
    nums = lambda v: tuple(int(x) for x in v.split(".")[:2] if x.isdigit())
    return nums(have) < nums(want)

if _older(_version("transformers"), "4.49") or _version("accelerate") == "0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.49", "accelerate"], check=True)

import torch
import transformers
print(f"✓ torch {torch.__version__}, transformers {transformers.__version__}, python {sys.version.split()[0]}")
print(f"  GPUs: {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")
assert torch.cuda.device_count() >= 2, "Pick GPU T4 x2: the 7B in fp16 needs both T4s"


✓ torch 2.11.0+cu130, transformers 5.20.2, python 3.13.16
  GPUs: ['Tesla T4', 'Tesla T4']


In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import glob, json, math, os, random, re, shutil, time
from pathlib import Path

import numpy as np
from PIL import Image

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"
TEACHER_MODEL_ID = "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP"   # L0 step 2's 7B, kept 28 Oct (public)
TOPK = 20                      # tokens kept per answer position
BATCH = 8                      # questions per forward pass; halved automatically on out-of-memory
SHARD = 500                    # questions per file written
SMOKE_TEST = False             # True = 16 questions, to check the setup and read the soft labels
TIME_LIMIT_HOURS = 11.0        # stop cleanly before Kaggle's 12-hour limit
SEED = 42
OUTPUT_DIR = Path("/kaggle/working/L2S1b")
SHARD_DIR = OUTPUT_DIR / "shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)

def hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")

# ── The rules all L2 step 2 notebooks share (the same text in each) ─────────
import hashlib

PATH_SUFFIXES = (".png", ".jpg", ".jpeg", ".tif", ".tiff")
CHECKABLE = {"Building Damage Counting", "Road Damage Counting", "Disaster Type Recognition",
             "Disaster Bearing Bodies Recognition", "Disaster Scene Recognition", "relational reasoning"}
MAX_PIXELS = 768 * 28 * 28       # the 3B's image cap: a 1024 px image reaches it at 756 px (729 tokens); also the soft labels'
HELD_OUT_EVERY = 20              # 1 after-photo in 20 is held out to choose between the two 3Bs (fixed 8 Nov)
ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT = 55764, 38271, 1962   # counted on the PC from train_release.json, 8 Nov

def name_of(path):
    return Path(str(path).replace("\\", "/")).name

def scene(e):
    """The after-photo a question is about. A relational question's boxed picture, '<after-photo>_<k>.png',
    counts as its after-photo, so the same scene never sits on both sides of the split."""
    if e.get("post_image_path"):
        return name_of(e["post_image_path"])
    return re.sub(r"_\d+(\.png)$", r"\1", name_of(e.get("image_path", "")))

def held_out(e):
    return int(hashlib.md5(scene(e).encode()).hexdigest(), 16) % HELD_OUT_EVERY == 0

def resolve_image_path(rel_path):
    """L2 step 1's rule: a manifest path -> its file in the mirror (None if missing)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for candidate in (DATA_ROOT / rel_path, DATA_ROOT / "train_images" / rel_path, DATA_ROOT / "box_train_images" / rel_path,
                      DATA_ROOT / "masks" / rel_path, DATA_ROOT / "train_images" / "train_images" / filename,
                      DATA_ROOT / "box_train_images" / "box_train_images" / filename, DATA_ROOT / "masks" / "masks" / filename,
                      Path("/tmp") / rel_path):
        if candidate.exists():
            return str(candidate)
    return None

def answer_text(entry):
    ans = entry.get("training_answer", "")
    if isinstance(ans, list):
        ans = ans[0] if ans else ""
    return str(ans).strip()

def eligible(entry):
    """L2 step 1's rule (the 7B's training rule): no segmentation, a text answer, and every image found."""
    if entry.get("task") == "Referring Expression Segmentation":
        return False
    ans = answer_text(entry)
    if not ans or ans.lower().endswith(PATH_SUFFIXES):
        return False
    if "image_path" in entry:
        return resolve_image_path(entry["image_path"]) is not None
    return (resolve_image_path(entry.get("pre_image_path", "")) is not None
            and resolve_image_path(entry.get("post_image_path", "")) is not None)

def smart_resize(height, width, min_pixels, max_pixels, factor=28):
    """Qwen2.5-VL's rule for the size an image reaches the model at (its image processor's smart_resize)."""
    h, w = round(height / factor) * factor, round(width / factor) * factor
    if h * w > max_pixels:
        beta = math.sqrt(height * width / max_pixels)
        h, w = math.floor(height / beta / factor) * factor, math.floor(width / beta / factor) * factor
    elif h * w < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h, w = math.ceil(height * beta / factor) * factor, math.ceil(width * beta / factor) * factor
    return h, w

def sized(img, max_pixels=MAX_PIXELS):
    """The image at the 3B's cap: only an image above it is scaled down (as L0 did at full size)."""
    if img.width * img.height <= max_pixels:
        return img
    h, w = smart_resize(img.height, img.width, 56 * 56, max_pixels)
    return img.resize((w, h), Image.BICUBIC)

assert smart_resize(1024, 1024, 56 * 56, MAX_PIXELS) == (756, 756)
assert held_out({"image_path": "x\\box_train_images\\a_post_7_3.png"}) == held_out({"post_image_path": "a_post_7.png"})

print(f"Teacher: {TEACHER_MODEL_ID}; top {TOPK} per answer token; image cap {MAX_PIXELS:,} pixels"
      + ("  [SMOKE TEST: 16 questions]" if SMOKE_TEST else ""))


Teacher: AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP; top 20 per answer token; image cap 602,112 pixels


In [3]:
# ── Cell 3: Load the 7B in fp16 across both T4s, and check the image size ─────
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

processor = AutoProcessor.from_pretrained(TEACHER_MODEL_ID, token=hf_token())
processor.tokenizer.padding_side = "right"            # the answer positions are found by their tokens, not by padding
dtype_arg = "torch_dtype" if _older(transformers.__version__, "4.56") else "dtype"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(TEACHER_MODEL_ID, **{dtype_arg: torch.float16}, device_map="auto",
                                                           attn_implementation="sdpa", token=hf_token())
model.eval()
tok = processor.tokenizer
MARKER = tok("<|im_start|>assistant\n", add_special_tokens=False)["input_ids"]
IM_END = tok.convert_tokens_to_ids("<|im_end|>")
_, h, w = processor.image_processor(images=[sized(Image.new("RGB", (1024, 1024), (90, 120, 80)))],
                                    return_tensors="pt")["image_grid_thw"][0].tolist()
assert (h * 14, w * 14) == (756, 756), f"A 1024 px image would reach the 7B at {h * 14} x {w * 14} px, not 756"
print(f"✓ 7B loaded; a 1024 px image reaches it at 756 px; GPU memory "
      f"{[f'{torch.cuda.memory_allocated(i) / 1e9:.1f} GB' for i in range(torch.cuda.device_count())]}")


processor_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/421 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 16.6GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

✓ 7B loaded; a 1024 px image reaches it at 756 px; GPU memory ['8.5 GB', '8.1 GB']


In [4]:
# ── Cell 4: The questions: checkable, not held out ─────────────────────────────
with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)
eligible_entries = [e for e in raw_data if eligible(e)]
TODO_ALL = [i for i, e in enumerate(eligible_entries) if e["task"] in CHECKABLE and not held_out(e)]
n_held = sum(1 for e in eligible_entries if e["task"] in CHECKABLE and held_out(e))
assert (len(eligible_entries), len(TODO_ALL), n_held) == (ELIGIBLE, CHECKABLE_TRAIN, CHECKABLE_HELD_OUT), \
    f"{len(eligible_entries):,} eligible, {len(TODO_ALL):,} to do, {n_held:,} held out: not the counts fixed on 8 Nov"
if SMOKE_TEST:      # the longest answer (memory) and 15 others, one relational (one image) among them
    longest = max(TODO_ALL, key=lambda i: len(answer_text(eligible_entries[i])))
    rel = next(i for i in TODO_ALL if eligible_entries[i]["task"] == "relational reasoning")
    rest = random.Random(SEED).sample([i for i in TODO_ALL if i not in (longest, rel)], 14)
    TODO_ALL = sorted([longest, rel] + rest)

def messages_of(entry):
    """The images first, then the training prompt, then the label as the answer: what the 3B will train on."""
    keys = ["image_path"] if "image_path" in entry else ["pre_image_path", "post_image_path"]
    images = [sized(Image.open(resolve_image_path(entry[k])).convert("RGB")) for k in keys]
    prompt = entry.get("prompts", "")
    prompt = prompt[0] if isinstance(prompt, list) else prompt
    return [{"role": "user", "content": [{"type": "image"} for _ in images] + [{"type": "text", "text": prompt}]},
            {"role": "assistant", "content": [{"type": "text", "text": answer_text(entry)}]}], images

def answer_span(ids):
    """(first answer position, position of the end-of-answer token) in one row of token ids."""
    for start in range(len(ids) - len(MARKER), -1, -1):
        if ids[start:start + len(MARKER)] == MARKER:
            first = start + len(MARKER)
            return first, ids.index(IM_END, first)
    raise ValueError("no assistant turn in the row")

_ids = MARKER + [5, 6, 7, IM_END, 9]
assert answer_span([1, 2] + _ids) == (2 + len(MARKER), 2 + len(MARKER) + 3)
print(f"✓ {len(TODO_ALL):,} questions ({n_held:,} held out, never shown to the 7B here); by task: "
      f"{ {t: sum(eligible_entries[i]['task'] == t for i in TODO_ALL) for t in sorted(CHECKABLE)} }")


✓ 38,271 questions (1,962 held out, never shown to the 7B here); by task: {'Building Damage Counting': 13849, 'Disaster Bearing Bodies Recognition': 7381, 'Disaster Scene Recognition': 6742, 'Disaster Type Recognition': 1543, 'Road Damage Counting': 6974, 'relational reasoning': 1782}


In [5]:
# ── Cell 5: The soft labels, written in shards as they come (resumable) ──────
from tqdm.auto import tqdm

previous = sorted(glob.glob("/kaggle/input/**/L2S1b/shards/*.npz", recursive=True))
for p in previous:                                    # continue a stopped version: copy its shards in
    if not (SHARD_DIR / Path(p).name).exists():
        shutil.copy(p, SHARD_DIR / Path(p).name)
done = set()
for p in sorted(SHARD_DIR.glob("*.npz")):
    with np.load(p) as z:
        done.update(z["entry_idx"].tolist())
todo = [i for i in TODO_ALL if i not in done]
print(f"{len(done):,} questions already done{' (from ' + str(len(previous)) + ' earlier shards)' if previous else ''}; "
      f"{len(todo):,} to do")

@torch.inference_mode()
def soft_labels(idxs):
    """One forward pass over a batch -> per question (entry index, target ids, top-k ids, top-k log-probs, target log-probs)."""
    convs = [messages_of(eligible_entries[i]) for i in idxs]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m, _ in convs]
    inputs = processor(text=texts, images=[im for _, ims in convs for im in ims], padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)
    hidden = model.model(**inputs).last_hidden_state
    spans, rows = [], []
    for b, i in enumerate(idxs):
        ids = inputs["input_ids"][b].tolist()
        first, end = answer_span(ids)
        spans.append((i, ids[first:end + 1]))
        rows.append(hidden[b, first - 1:end])           # the position before each target token predicts it
    head = model.lm_head
    logp = torch.log_softmax(head(torch.cat(rows).to(head.weight.device)).float(), -1)
    targets = torch.tensor([t for _, ids in spans for t in ids], device=logp.device)
    top_v, top_i = logp.topk(TOPK, dim=-1)
    target_logp = logp.gather(1, targets[:, None])[:, 0]
    out, k = [], 0
    for i, ids in spans:
        n = len(ids)
        out.append((i, np.array(ids, np.int32), top_i[k:k + n].cpu().numpy().astype(np.int32),
                    top_v[k:k + n].cpu().numpy().astype(np.float16), target_logp[k:k + n].cpu().numpy().astype(np.float16)))
        k += n
    return out

def soft_labels_safe(idxs):
    """Halves the batch on out-of-memory instead of stopping the run."""
    try:
        return soft_labels(idxs)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(idxs) == 1:
            raise
        half = len(idxs) // 2
        return soft_labels_safe(idxs[:half]) + soft_labels_safe(idxs[half:])

def write_shard(rows):
    offsets = np.cumsum([0] + [len(r[1]) for r in rows]).astype(np.int64)
    name = SHARD_DIR / f"shard_{len(list(SHARD_DIR.glob('*.npz'))):05d}_{rows[0][0]}.npz"
    np.savez(name, entry_idx=np.array([r[0] for r in rows], np.int32), offsets=offsets,
             target_ids=np.concatenate([r[1] for r in rows]), topk_ids=np.concatenate([r[2] for r in rows]),
             topk_logp=np.concatenate([r[3] for r in rows]), target_logp=np.concatenate([r[4] for r in rows]))

mismatch, buffer, shown = [], [], 0
deadline, t0, stopped = time.time() + TIME_LIMIT_HOURS * 3600, time.time(), False
for start in tqdm(range(0, len(todo), BATCH), desc="Soft labels"):
    if time.time() > deadline:
        stopped = True
        break
    for i, ids, top_i, top_v, t_lp in soft_labels_safe(todo[start:start + BATCH]):
        label = answer_text(eligible_entries[i])
        if tok.decode(ids[:-1].tolist()).strip() != label.strip():
            mismatch.append(i)
        if shown < 3:                                  # read a few before trusting the run
            words = [tok.decode([t]) for t in ids[:8]]
            print(f"\n[{eligible_entries[i]['task']}] label: {label[:100]!r}\n  first tokens and the 7B's top 3 there:")
            for w, cand, val in zip(words, top_i[:8], top_v[:8]):
                print(f"    {w!r:>14}: " + ", ".join(f"{tok.decode([int(c)])!r} {math.exp(float(v)):.2f}"
                                                     for c, v in zip(cand[:3], val[:3])))
            shown += 1
        buffer.append((i, ids, top_i, top_v, t_lp))
    if len(buffer) >= SHARD:
        write_shard(buffer)
        buffer = []
    if len(mismatch) > max(5, 0.005 * (start + BATCH)):
        raise RuntimeError(f"{len(mismatch)} answers don't decode to their label (e.g. entry {mismatch[0]}): "
                           "the answer positions are wrong, so the soft labels would be misaligned")
if buffer:
    write_shard(buffer)
RUN_HOURS = (time.time() - t0) / 3600
print(f"\n{'Time limit reached: stopped cleanly' if stopped else 'Pass complete'} after {RUN_HOURS:.2f} h; "
      f"{len(mismatch)} label mismatches")


0 questions already done; 38,271 to do


Soft labels:   0%|          | 0/4784 [00:00<?, ?it/s]


[Disaster Bearing Bodies Recognition] label: 'The essential land-cover objects that appear damaged in this disaster zone include buildings, roads,'
  first tokens and the 7B's top 3 there:
             'The': 'The' 1.00, 'Both' 0.00, 'In' 0.00
      ' essential': ' essential' 0.87, ' damaged' 0.09, ' disaster' 0.02
           ' land': ' land' 1.00, ' damaged' 0.00, 'land' 0.00
          '-cover': '-cover' 1.00, ' cover' 0.00, '-covered' 0.00
        ' objects': ' objects' 1.00, ' features' 0.00, ' types' 0.00
           ' that': ' that' 0.99, ' identified' 0.01, ' observed' 0.00
         ' appear': ' appear' 0.82, ' have' 0.12, ' show' 0.05
        ' damaged': ' to' 0.65, ' damaged' 0.35, 'dam' 0.00

[Disaster Bearing Bodies Recognition] label: 'The damaged objects in the affected area include buildings, roads, forest, and farmland.'
  first tokens and the 7B's top 3 there:
             'The': 'The' 1.00, 'A' 0.00, 'Both' 0.00
        ' damaged': ' categories' 0.64, ' damaged' 0.17, '

In [6]:
# ── Cell 6: What the soft labels say, the projection, and the output files ────
import pandas as pd

ent, tgt, top_i, tlp, offs = [], [], [], [], []
for p in sorted(SHARD_DIR.glob("*.npz")):
    with np.load(p) as z:
        ent.append(z["entry_idx"])
        offs.append(np.diff(z["offsets"]))
        tgt.append(z["target_ids"])
        top_i.append(z["topk_ids"][:, 0])
        tlp.append(z["target_logp"].astype(np.float32))
ent, lens, tgt, top1, tlp = (np.concatenate(x) for x in (ent, offs, tgt, top_i, tlp))
assert len(set(ent.tolist())) == len(ent), "a question appears in two shards"
token_task = np.repeat([eligible_entries[i]["task"] for i in ent], lens)
rows = []
for task in sorted(set(token_task)):
    sel = token_task == task
    rows.append({"task": task, "questions": int(sum(eligible_entries[i]["task"] == task for i in ent)),
                 "answer_tokens": int(sel.sum()), "top1_is_label_%": round(100 * float((top1[sel] == tgt[sel]).mean()), 1),
                 "mean_label_prob": round(float(np.exp(tlp[sel]).mean()), 3)})
by_task = pd.DataFrame(rows)
by_task.to_csv(OUTPUT_DIR / "by_task.csv", index=False)
print(by_task.to_string(index=False))
complete = len(ent) >= CHECKABLE_TRAIN
per_q = RUN_HOURS * 3600 / max(len(todo), 1)
projected = per_q * CHECKABLE_TRAIN / 3600
print(f"\n{len(ent):,} of {CHECKABLE_TRAIN:,} questions done{' (complete)' if complete else ''}; "
      f"{per_q:.2f} s per question this session; all {CHECKABLE_TRAIN:,} would take about {projected:.1f} h")

try:
    from huggingface_hub import model_info
    teacher_commit = model_info(TEACHER_MODEL_ID, token=hf_token()).sha
except Exception:
    teacher_commit = None
config = {"task": "L2 step 1b: soft labels from the new 7B", "smoke_test": SMOKE_TEST, "teacher": TEACHER_MODEL_ID,
          "teacher_commit": teacher_commit, "topk": TOPK, "max_pixels": MAX_PIXELS, "answer": "the dataset's label (training_answer)",
          "positions": "every answer token and the end-of-answer token", "questions_done": len(ent), "complete": complete,
          "held_out_every": HELD_OUT_EVERY, "label_mismatches": len(mismatch), "seconds_per_question": round(per_q, 3),
          "projected_hours_all": round(projected, 1), "versions": {"torch": torch.__version__, "transformers": transformers.__version__}}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUTPUT_DIR / "README.md").write_text("\n".join([
    "# L2 step 1b: soft labels from the new 7B" + (" (SMOKE TEST)" if SMOKE_TEST else ""), "",
    f"The 7B `{TEACHER_MODEL_ID}` read each checkable training question with its label as the answer, at the 3B's image size "
    f"(at most {MAX_PIXELS:,} pixels). At each answer token and at the end-of-answer token, the 20 most likely tokens are kept with "
    f"their log-probabilities. {len(ent):,} of {CHECKABLE_TRAIN:,} questions done; the 1,962 on held-out after-photos are left out.", "",
    "## By task", "", "| " + " | ".join(by_task.columns) + " |", "|" + "---|" * len(by_task.columns),
    *["| " + " | ".join(str(v) for v in r) + " |" for r in by_task.itertuples(index=False)], "",
    "- `top1_is_label_%`: answer tokens where the 7B's most likely token is the label's.",
    "- `mean_label_prob`: the 7B's mean probability of the label's token.", "",
    "## Files", "",
    "- `shards/*.npz`: per question `entry_idx` (its index among the 55,764 eligible entries), `offsets`, and per answer token "
    "`target_ids`, `topk_ids` and `topk_logp` (the top 20), `target_logp`.",
    "- `by_task.csv`, `config.json`.",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUTPUT_DIR}: README.md, config.json, by_task.csv, {len(list(SHARD_DIR.glob('*.npz')))} shards")
if not complete:
    print("NOTE: not complete. Attach this version's output and run again: Cell 5 continues from its shards.")


                               task  questions  answer_tokens  top1_is_label_%  mean_label_prob
           Building Damage Counting       9019         147701             86.7            0.835
Disaster Bearing Bodies Recognition       7381         162383             90.8            0.872

16,400 of 38,271 questions done; 1.04 s per question this session; all 38,271 would take about 11.0 h
✓ Wrote /kaggle/working/L2S1b: README.md, config.json, by_task.csv, 33 shards
NOTE: not complete. Attach this version's output and run again: Cell 5 continues from its shards.
